In [1]:
# ================================================================
# NOTEBOOK : nb_gold_product_rank
# Reads    : silver_lakehouse → silver_sales, silver_product
# Writes   : gold_lakehouse  → gold_product_rankings
# Logic    : Product-level performance — revenue, margin, rank
# ================================================================

StatementMeta(, 5d024db0-7dae-42b0-bba1-b4b1b9a90847, 3, Finished, Available, Finished, False)

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col,round
from pyspark.sql.window import Window

sales = spark.sql("SELECT * FROM silver_lakehouse.dbo.silver_sales")
product = spark.sql("SELECT * FROM silver_lakehouse.dbo.silver_product")

# ── Aggregate at product grain ─────────────────────────────────────

product_agg = ( sales
.filter(col("IsReturn") == False)
.groupBy("ProductID").
agg(F.sum("TotalAmount").alias("TotalRevenue"),
F.sum("Quantity").alias("UnitsSold"),
F.count("TransactionID").alias("TransactionCount"),
F.countDistinct("StoreID").alias("StoresSoldIn"))
)


# ── Join product dimension ─────────────────────────────────────────
gold_df = product_agg.join(product, on = "ProductID", how = "left")


# ── Margin & ranking ─────────────────────────────────────────────

gold_df = (gold_df.withColumn("EstimatedProfit", round(col("UnitsSold") * (col("ListPrice") - col("CostPrice")),2))
.withColumn("CategoryRank", F.rank().over(Window.partitionBy("Category").orderBy(F.desc("TotalRevenue"))))
.withColumn("OverallRank", F.rank().over(Window.orderBy(F.desc("TotalRevenue"))))
.withColumn("_gold_load_ts",F.current_timestamp() )
)

gold_df.write.format("delta").mode("overwrite")\
.option("overwriteSchema", "true")\
.saveAsTable("gold_product_rankings")


print("[DONE] gold_product_rankings:{gold_df.count()} rows")
display(gold_df.orderBy("OverallRank").limit(10))



StatementMeta(, 5d024db0-7dae-42b0-bba1-b4b1b9a90847, 6, Finished, Available, Finished, False)

[DONE] gold_product_rankings:{gold_df.count()} rows


SynapseWidget(Synapse.DataFrame, dcf8d02d-da5b-4b83-909b-c44fa2cdd749)